In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

# ============================================================
# Fourier Time Embedding (unchanged)
# ============================================================


class GaussianFourierProjection(nn.Module):
    def __init__(self, embedding_size=256, scale=1.0):
        super().__init__()
        self.W = nn.Parameter(torch.randn(embedding_size) * scale)

    def forward(self, t):
        if t.dim() == 1:
            t = t.unsqueeze(1)
        t_proj = t * self.W[None, :] * 2 * np.pi
        return torch.cat([torch.sin(t_proj), torch.cos(t_proj)], dim=-1)


class TimeEmbedding(nn.Module):
    def __init__(self, t_features, hidden_dim, T):
        super().__init__()
        self.fourier = GaussianFourierProjection(t_features, scale=1 / T)
        self.mlp = nn.Sequential(
            nn.Linear(2 * t_features, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t):
        return self.mlp(self.fourier(t))


# ============================================================
# BigGAN++ style 1D Residual Block
# ============================================================


class ResnetBlock1DBigGANpp(nn.Module):
    def __init__(
        self,
        channels,
        t_emb_dim=None,
        out_channels=None,
        up=False,
        down=False,
        dropout=0.1,
        skip_rescale=True,
        init_scale=0.0,
        groups=8,
    ):
        super().__init__()
        out_channels = out_channels or channels
        self.up = up
        self.down = down
        self.skip_rescale = skip_rescale

        # Normalization layers
        self.norm1 = nn.GroupNorm(groups, channels, eps=1e-6)
        self.norm2 = nn.GroupNorm(groups, out_channels, eps=1e-6)

        # Convolutions
        self.conv1 = nn.Conv1d(channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)
        if channels != out_channels or up or down:
            self.skip_conv = nn.Conv1d(channels, out_channels, kernel_size=1)
        else:
            self.skip_conv = None

        # FiLM projection from time embedding
        if t_emb_dim is not None:
            self.t_proj = nn.Linear(t_emb_dim, out_channels * 2)
            nn.init.zeros_(self.t_proj.bias)

        self.act = nn.SiLU()
        self.dropout = nn.Dropout(dropout)

        # --- FIX: init_scale was accepted but never used. BigGAN++/NCSN++
        # zero- (or near-zero-) initialize the *second* conv in the block so
        # each residual block starts close to an identity map, which
        # stabilizes early training of deep score networks.
        if init_scale == 0.0:
            nn.init.zeros_(self.conv2.weight)
        else:
            nn.init.normal_(self.conv2.weight, std=init_scale)
        nn.init.zeros_(self.conv2.bias)

    def forward(self, x, t_emb=None):
        h = self.act(self.norm1(x))

        # Optional up/downsampling
        if self.up:
            h = F.interpolate(h, scale_factor=2, mode="nearest")
            x = F.interpolate(x, scale_factor=2, mode="nearest")
        elif self.down:
            h = F.avg_pool1d(h, kernel_size=2, stride=2)
            x = F.avg_pool1d(x, kernel_size=2, stride=2)

        # First convolution
        h = self.conv1(h)

        # FiLM conditioning
        if t_emb is not None:
            gamma_beta = self.t_proj(self.act(t_emb))
            gamma, beta = gamma_beta.chunk(2, dim=1)
            gamma = gamma[:, :, None]
            beta = beta[:, :, None]
            h = h * (1 + gamma) + beta

        # Second normalization, activation, dropout, conv
        h = self.conv2(self.dropout(self.act(self.norm2(h))))

        # Skip connection
        if self.skip_conv is not None:
            x = self.skip_conv(x)

        # Optional rescale
        if self.skip_rescale:
            return (x + h) / math.sqrt(2.0)
        else:
            return x + h


# ============================================================
# DownBlock, UpBlock, Bottleneck (unchanged)
# ============================================================


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_emb_dim):
        super().__init__()
        self.res = ResnetBlock1DBigGANpp(in_ch, t_emb_dim)
        self.down = nn.Conv1d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)

    def forward(self, x, t_emb):
        x = self.res(x, t_emb)
        skip = x
        x = self.down(x)
        return x, skip


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, t_emb_dim):
        super().__init__()
        self.res = ResnetBlock1DBigGANpp(in_ch + skip_ch, t_emb_dim)
        self.conv = nn.Conv1d(in_ch + skip_ch, out_ch, kernel_size=3, padding=1)

    def match_length(self, x, target):
        diff = target.shape[-1] - x.shape[-1]
        if diff > 0:
            x = F.pad(x, (0, diff))
        elif diff < 0:
            x = x[..., : target.shape[-1]]
        return x

    def forward(self, x, skip, t_emb):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        x = self.match_length(x, skip)
        x = torch.cat([x, skip], dim=1)
        x = self.res(x, t_emb)
        x = self.conv(x)
        return x


class Bottleneck(nn.Module):
    def __init__(self, channels, t_emb_dim):
        super().__init__()
        self.res1 = ResnetBlock1DBigGANpp(channels, t_emb_dim)
        self.attn = AttentionBlock(channels)
        self.res2 = ResnetBlock1DBigGANpp(channels, t_emb_dim)

    def forward(self, x, t_emb):
        x = self.res1(x, t_emb)
        x = self.attn(x)
        x = self.res2(x, t_emb)
        return x


class AttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.attn = nn.MultiheadAttention(
            channels,
            num_heads=num_heads,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(channels)
        self.ff = nn.Sequential(
            nn.Linear(channels, channels * 4),
            nn.SiLU(),
            nn.Linear(channels * 4, channels),
        )

    def forward(self, x):
        x_t = x.transpose(1, 2)  # [B, L, C]

        h = self.norm1(x_t)
        attn_out, _ = self.attn(h, h, h)
        x_t = x_t + attn_out

        h = self.norm2(x_t)
        x_t = x_t + self.ff(h)

        return x_t.transpose(1, 2)


# ============================================================
# FULL UNET
# ============================================================


class UNet(nn.Module):
    """
    Predicts a K-channel complex output given:
      - X: [B, K, L] complex   -- e.g. a K-channel conditioning spectrum
      - Y: [B, 1, L] complex   -- e.g. the single noisy/target channel
      - t: [B]                 -- diffusion timestep

    NOTE (fix): the original docstring claimed both X and Y were [B, K, L],
    but in_proj was sized for 2*(K+1) input channels, which only matches
    cat(X, Y) -> real/imag if Y has exactly 1 channel (K + 1 total complex
    channels -> 2*(K+1) real channels). That's the assumption now enforced
    explicitly below via the Y shape check, instead of silently crashing
    inside nn.Conv1d with an unhelpful shape error.

    If your actual data has Y with K channels too, change `y_channels=1`
    below to `y_channels=K` and in_proj will use 2*(2K) channels instead.
    """

    def __init__(
        self,
        K,
        base_channels=64,
        channel_mults=(1, 2, 2, 4, 2),
        t_features=256,
        T=1,
        y_channels=1,
    ):
        super().__init__()

        self.K = K
        self.y_channels = y_channels
        t_emb_dim = base_channels * 4

        self.time_embed = TimeEmbedding(t_features, t_emb_dim, T)

        # --- FIX: in_proj sized from explicit (K, y_channels) instead of
        # the previous hard-coded "K + 1" guess, so the channel count is
        # always consistent with what forward() actually concatenates.
        in_channels = 2 * (K + y_channels)  # *2 for real+imag
        self.in_proj = nn.Conv1d(
            in_channels,
            base_channels,
            kernel_size=1,
        )

        # ---------------- Down Path ----------------
        channels = [base_channels * m for m in channel_mults]

        self.down_blocks = nn.ModuleList()
        skip_channels = []

        in_ch = base_channels
        for ch in channels:
            self.down_blocks.append(DownBlock(in_ch, ch, t_emb_dim))
            skip_channels.append(in_ch)
            in_ch = ch

        # ---------------- Bottleneck ----------------
        self.bottleneck = Bottleneck(in_ch, t_emb_dim)

        # ---------------- Up Path ----------------
        self.up_blocks = nn.ModuleList()

        for ch, skip_ch in zip(reversed(channels), reversed(skip_channels)):
            self.up_blocks.append(UpBlock(in_ch, skip_ch, ch, t_emb_dim))
            in_ch = ch

        # ---------------- Output ----------------
        self.out_proj = nn.Conv1d(
            base_channels,
            2 * K,
            kernel_size=1,
        )

    def forward(self, x, t):
        """
        X: complex tensor [B, K, L]
        Y: complex tensor [B, y_channels, L]   (y_channels=1 by default)
        t: [B]
        """
        # --- FIX: fail loudly with a clear message instead of a cryptic
        # Conv1d channel-mismatch error if the caller passes the wrong
        # shapes for X / Y.
        assert X.shape[1] == self.K, (
            f"X has {X.shape[1]} channels, expected K={self.K}"
        )
        assert Y.shape[1] == self.y_channels, (
            f"Y has {Y.shape[1]} channels, expected y_channels={self.y_channels}"
        )
        assert torch.is_complex(X) and torch.is_complex(Y), "X and Y must be complex tensors"

        # Flatten complex to real channels
        XY = torch.cat([X, Y], dim=1)
        XY = torch.cat([XY.real, XY.imag], dim=1)

        x = self.in_proj(XY)
        t_emb = self.time_embed(t)

        skips = []
        for down in self.down_blocks:
            x, skip = down(x, t_emb)
            skips.append(skip)

        x = self.bottleneck(x, t_emb)

        for up, skip in zip(self.up_blocks, reversed(skips)):
            x = up(x, skip, t_emb)

        out = self.out_proj(x)

        # Convert back to complex
        r = out[:, : self.K, :]
        i = out[:, self.K :, :]
        return torch.complex(r, i)

In [3]:



def make_complex(B, C, L):
    return torch.randn(B, C, L, dtype=torch.float32) + 1j * torch.randn(
        B, C, L, dtype=torch.float32
    )


def test_forward_shape():
    """Basic forward pass: output shape must match input X's [B, K, L]."""
    B, K, L = 2, 4, 64
    model = UNet(K=K, base_channels=16, channel_mults=(1, 2, 2))

    X = make_complex(B, K, L)
    Y = make_complex(B, 1, L)  # y_channels defaults to 1
    t = torch.rand(B)

    out = model(X, Y, t)

    assert out.shape == (B, K, L), f"expected {(B, K, L)}, got {out.shape}"
    assert torch.is_complex(out), "output should be complex"
    print("test_forward_shape: PASS  ->", out.shape, out.dtype)


def test_backward_pass():
    """Gradients should flow through the whole network without NaNs/errors."""
    B, K, L = 2, 4, 64
    model = UNet(K=K, base_channels=16, channel_mults=(1, 2, 2))

    X = make_complex(B, K, L)
    Y = make_complex(B, 1, L)
    t = torch.rand(B)

    out = model(X, Y, t)
    loss = (out.real ** 2 + out.imag ** 2).mean()
    loss.backward()

    n_params_with_grad = sum(
        1 for p in model.parameters() if p.grad is not None
    )
    n_params_total = sum(1 for _ in model.parameters())
    has_nan = any(
        torch.isnan(p.grad).any() for p in model.parameters() if p.grad is not None
    )

    assert n_params_with_grad == n_params_total, (
        f"only {n_params_with_grad}/{n_params_total} params received gradients"
    )
    assert not has_nan, "NaNs found in gradients"
    print(f"test_backward_pass: PASS  -> {n_params_with_grad}/{n_params_total} params got grads, loss={loss.item():.4f}")


def test_variable_length_input():
    """Odd / non-power-of-two lengths should still work thanks to match_length."""
    B, K = 4, 2
    model = UNet(K=K, base_channels=16, channel_mults=(1, 2, 2))

    for L in [33, 50, 100]:
        X = make_complex(B, K, L)
        Y = make_complex(B, 1, L)
        t = torch.rand(B)
        out = model(X, Y, t)
        assert out.shape == (B, K, L), f"L={L}: expected {(B, K, L)}, got {out.shape}"
    print("test_variable_length_input: PASS  -> L in [33, 50, 100] all OK")


def test_custom_y_channels():
    """If Y genuinely has K channels (not 1), y_channels=K should still work."""
    B, K, L = 2, 4, 64
    model = UNet(K=K, base_channels=16, channel_mults=(1, 2, 2), y_channels=K)

    X = make_complex(B, K, L)
    Y = make_complex(B, K, L)
    t = torch.rand(B)

    out = model(X, Y, t)
    assert out.shape == (B, K, L)
    print("test_custom_y_channels: PASS  -> y_channels=K works when explicitly set")


def test_shape_assertions_fire():
    """Wrong channel counts should raise a clear assertion, not a cryptic Conv1d error."""
    B, K, L = 2, 4, 64
    model = UNet(K=K, base_channels=16, channel_mults=(1, 2, 2))

    X_wrong = make_complex(B, K + 1, L)  # wrong K
    Y = make_complex(B, 1, L)
    t = torch.rand(B)
    try:
        model(X_wrong, Y, t)
        raise SystemExit("expected AssertionError for wrong X channels, none raised")
    except AssertionError as e:
        assert "X has" in str(e)

    X = make_complex(B, K, L)
    Y_wrong = make_complex(B, 2, L)  # wrong y_channels
    try:
        model(X, Y_wrong, t)
        raise SystemExit("expected AssertionError for wrong Y channels, none raised")
    except AssertionError as e:
        assert "Y has" in str(e)

    print("test_shape_assertions_fire: PASS  -> bad shapes raise clear AssertionErrors")


def test_old_buggy_shapes_would_have_failed():
    """
    Regression check: confirms the original bug. With the *old* in_proj sizing
    (2*(K+1)) and X, Y BOTH at K channels (as the original docstring claimed),
    the concatenated channel count is 4K, which never equals 2*(K+1) for K>1.
    This documents why the fix (y_channels=1 default) was necessary.
    """
    K = 4
    old_in_proj_channels = 2 * (K + 1)
    actual_concat_channels_if_both_K = 2 * (K + K)
    assert old_in_proj_channels != actual_concat_channels_if_both_K
    print(
        "test_old_buggy_shapes_would_have_failed: PASS  -> "
        f"old in_proj expected {old_in_proj_channels} channels, "
        f"but X,Y both at K={K} would produce {actual_concat_channels_if_both_K} "
        "(confirms original bug)"
    )


if __name__ == "__main__":
    test_forward_shape()
    test_backward_pass()
    test_variable_length_input()
    test_custom_y_channels()
    test_shape_assertions_fire()
    test_old_buggy_shapes_would_have_failed()
    print("\nAll tests passed.")

test_forward_shape: PASS  -> torch.Size([2, 4, 64]) torch.complex64
test_backward_pass: PASS  -> 113/113 params got grads, loss=0.0625
test_variable_length_input: PASS  -> L in [33, 50, 100] all OK
test_custom_y_channels: PASS  -> y_channels=K works when explicitly set
test_shape_assertions_fire: PASS  -> bad shapes raise clear AssertionErrors
test_old_buggy_shapes_would_have_failed: PASS  -> old in_proj expected 10 channels, but X,Y both at K=4 would produce 16 (confirms original bug)

All tests passed.


In [21]:
import torch
import time

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# Problem size
# -----------------------
B, N = 4, 2000

dtype = torch.complex64

# -----------------------
# Data
# -----------------------
E = torch.randn(N, N, dtype=dtype, device=device)
s = torch.randn(B, N, dtype=dtype, device=device)
x = torch.randn(B, N, dtype=dtype, device=device)

E = E.contiguous()

# -----------------------
# Methods
# -----------------------

def slow():
    return E @ torch.diag_embed(s)

def fast():
    return E.unsqueeze(0) * s.unsqueeze(1)

def apply_slow_to_x(M):
    return (M @ x.unsqueeze(-1)).squeeze(-1)

def operator():
    tmp = torch.matmul(E.unsqueeze(0).transpose(1,2), x.unsqueeze(-1)).squeeze(-1)
    tmp = tmp * s
    return torch.matmul(E.unsqueeze(0), tmp.unsqueeze(-1)).squeeze(-1)

# -----------------------
# Timing helper
# -----------------------

def bench(fn, name, iters=50):
    # warmup
    for _ in range(10):
        _ = fn()

    if device == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    for _ in range(iters):
        out = fn()

    if device == "cuda":
        torch.cuda.synchronize()

    print(f"{name:<25}: {(time.perf_counter()-t0)/iters*1000:.4f} ms/iter")

    return out

# -----------------------
# 1. MATRIX ACCURACY TEST
# -----------------------

with torch.no_grad():
    A = slow()
    B_ = fast()

    diff = (A - B_).abs()

    print("\n=== MATRIX ERROR ===")
    print("max abs:", diff.max().item())
    print("mean abs:", diff.mean().item())
    print("rel error:", (diff / (A.abs() + 1e-8)).max().item())

# -----------------------
# 2. OPERATOR ACCURACY TEST
# -----------------------

with torch.no_grad():
    A_x = apply_slow_to_x(slow())
    B_x = apply_slow_to_x(fast())

    diff_x = (A_x - B_x).abs()

    print("\n=== APPLY ERROR ===")
    print("max abs:", diff_x.max().item())
    print("mean abs:", diff_x.mean().item())
    print("rel error:", (diff_x / (A_x.abs() + 1e-8)).max().item())

# -----------------------
# 3. TIMING
# -----------------------

print("\n=== TIMING ===")
bench(slow, "slow (diag + matmul)")
bench(fast, "fast (broadcast)")


=== MATRIX ERROR ===
max abs: 4.76837158203125e-07
mean abs: 1.13102851528879e-08
rel error: 1.1920224096684251e-07

=== APPLY ERROR ===
max abs: 1.9073486328125e-05
mean abs: 4.2296383071516175e-06
rel error: 1.5484716641367413e-05

=== TIMING ===
slow (diag + matmul)     : 27.0726 ms/iter
fast (broadcast)         : 0.3183 ms/iter


tensor([[[-2.6036e-02+1.3517e-01j, -1.0242e+00+1.3304e+00j,
           6.3666e-01-4.5411e-01j,  ...,
           1.3154e+00+4.2026e-01j,  9.3775e-01-1.2151e+00j,
           7.9626e-01+2.9298e+00j],
         [ 3.1029e-02-1.8184e-01j,  7.6800e-01-8.3969e-01j,
           8.2665e-02-1.1247e+00j,  ...,
          -1.1251e+00-1.1635e+00j, -1.3908e-01-6.4249e-01j,
          -1.0967e+00+6.1149e-01j],
         [ 2.5173e-01+1.3614e-01j, -1.8439e-01+8.2670e-02j,
           1.3152e-01-1.4896e-01j,  ...,
           6.8906e-01+4.6689e-01j,  8.3788e-01+2.7650e-01j,
          -2.2703e-01+7.6652e-01j],
         ...,
         [ 1.3930e-01-1.2183e-01j, -1.8620e-01-6.9023e-01j,
           1.8542e+00-1.4968e+00j,  ...,
           1.5382e-01+3.9266e-02j,  4.8986e-01-3.5522e-02j,
           2.7683e-01+9.3245e-01j],
         [-5.2383e-02-3.4252e-03j, -7.2668e-01+5.8678e-02j,
           5.0166e-01+6.0006e-01j,  ...,
           1.2085e+00-1.1849e-01j, -1.2945e+00+5.9636e-02j,
          -1.3959e+00+3.1909e-01j],
 

In [23]:
import torch
import time

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# Sizes
# -----------------------
B, N = 4, 2000

dtype = torch.complex64

# -----------------------
# Fake spectral system
# -----------------------
V = torch.randn(N, N, dtype=dtype, device=device)
V_inv = torch.linalg.inv(V)

t = torch.randn(B, N, dtype=dtype, device=device)

eig = torch.randn(B, N, dtype=dtype, device=device)

# simulate your function output
exp_f = torch.exp(eig)

# -----------------------
# ORIGINAL METHOD
# -----------------------
def original():
    V_scaled = V.unsqueeze(0) * exp_f.unsqueeze(1)
    return V_scaled @ V_inv

# -----------------------
# NEW METHOD (your rewritten version)
# -----------------------
def optimized():
    V_scaled = V * exp_f.unsqueeze(1)
    return torch.matmul(V_scaled, V_inv)

# -----------------------
# timing helper
# -----------------------
def bench(fn, name, iters=100):
    # warmup
    for _ in range(10):
        _ = fn()

    if device == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    for _ in range(iters):
        out = fn()

    if device == "cuda":
        torch.cuda.synchronize()

    print(f"{name:<20}: {(time.perf_counter()-t0)/iters*1000:.4f} ms/iter")
    return out

# -----------------------
# RUN
# -----------------------
with torch.no_grad():

    A = bench(original, "original (unsqueeze)")
    B_ = bench(optimized, "optimized (broadcast)")

    # -----------------------
    # ERROR CHECK
    # -----------------------
    diff = (A - B_).abs()

    print("\n=== ERROR ===")
    print("max abs:", diff.max().item())
    print("mean abs:", diff.mean().item())
    print("rel error:", (diff / (A.abs() + 1e-8)).max().item())

original (unsqueeze): 26.6889 ms/iter
optimized (broadcast): 26.7224 ms/iter

=== ERROR ===
max abs: 0.0
mean abs: 0.0
rel error: 0.0


In [26]:
import torch
import time

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# CONFIG
# -----------------------
B, N = 4, 1024
dtype = torch.complex64

# -----------------------
# SYNTHETIC DATA
# -----------------------
V = torch.randn(N, N, dtype=dtype, device=device)
V_inv = torch.linalg.inv(V)

x = torch.randn(B, N, dtype=dtype, device=device)
eig = torch.randn(B, N, dtype=dtype, device=device)

# -----------------------
# OLD METHOD (FULL MATRIX)
# F = V diag(exp(eig)) V^{-1}
# -----------------------
def old_method():
    exp_f = torch.exp(eig)  # (B, N)

    # (B, N, N)
    V_scaled = V.unsqueeze(0) * exp_f.unsqueeze(1)

    # (B, N, N)
    F = torch.matmul(V_scaled, V_inv.unsqueeze(0))

    # (B, N)
    return torch.matmul(F, x.unsqueeze(-1)).squeeze(-1)

# -----------------------
# NEW METHOD (OPERATOR FORM)
# F(t)x = V (exp ⊙ (V^{-1}x))
# -----------------------
def new_method():
    exp_f = torch.exp(eig)  # (B, N)

    # (B, N)
    x_eig = torch.matmul(
        V_inv.unsqueeze(0),
        x.unsqueeze(-1)
    ).squeeze(-1)

    # elementwise scaling
    x_eig = x_eig * exp_f

    # back transform
    out = torch.matmul(
        V.unsqueeze(0),
        x_eig.unsqueeze(-1)
    ).squeeze(-1)

    return out

# -----------------------
# BENCHMARK HELPER
# -----------------------
def bench(fn, name, iters=100):
    # warmup
    for _ in range(10):
        _ = fn()

    if device == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    for _ in range(iters):
        out = fn()

    if device == "cuda":
        torch.cuda.synchronize()

    avg_time = (time.perf_counter() - t0) / iters * 1000
    print(f"{name:<25}: {avg_time:.4f} ms/iter")

    return out

# -----------------------
# COMPLEX ERROR METRICS
# -----------------------
def error(a, b):
    diff = a - b

    abs_err = diff.abs().max().item()
    mean_err = diff.abs().mean().item()
    rel_err = (diff.abs() / (a.abs() + 1e-8)).max().item()

    # phase-aware stability check
    cos_sim = torch.nn.functional.cosine_similarity(
        a.real, b.real, dim=0
    ).mean().item()

    return abs_err, mean_err, rel_err, cos_sim

# -----------------------
# RUN TEST
# -----------------------
with torch.no_grad():

    print("\n=== TIMING ===")
    out_old = bench(old_method, "OLD (full matrix)")
    out_new = bench(new_method, "NEW (operator)")

    print("\n=== ERROR CHECK ===")
    abs_err, mean_err, rel_err, cos_sim = error(out_old, out_new)

    print(f"max abs error : {abs_err:.6e}")
    print(f"mean abs error: {mean_err:.6e}")
    print(f"rel error     : {rel_err:.6e}")
    print(f"cosine sim    : {cos_sim:.6f}")


=== TIMING ===
OLD (full matrix)        : 4.2103 ms/iter
NEW (operator)           : 0.0709 ms/iter

=== ERROR CHECK ===
max abs error : 2.905187e-04
mean abs error: 6.036343e-05
rel error     : 3.832745e-05
cosine sim    : 1.000000


In [14]:
import torch

torch.manual_seed(0)

B, N = 4, 4000

E = torch.randn(N, N, dtype=torch.complex64)
s = torch.randn(B, N, dtype=torch.complex64)

# slow
slow = E @ torch.diag_embed(s)

# fast
fast = E.unsqueeze(0) * s.unsqueeze(1)

diff = (slow - fast).abs()

print("max error:", diff.max().item())
print("mean error:", diff.mean().item())

max error: 0.0
mean error: 0.0


In [4]:
"""
test_unet.py
============
Sanity-check suite for the enhanced UNet.
Run with:  python test_unet.py
"""

import sys
import traceback
import torch



DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PASSED, FAILED = [], []

def run(name, fn):
    try:
        fn()
        print(f"[PASS] {name}")
        PASSED.append(name)
    except Exception as e:
        print(f"[FAIL] {name}: {e}")
        traceback.print_exc()
        FAILED.append(name)


# ── component tests ──────────────────────────────────────────────────────────

def test_fourier_projection():
    m = GaussianFourierProjection(embedding_size=64, scale=1.0).to(DEVICE)
    out = m(torch.rand(8).to(DEVICE))
    assert out.shape == (8, 128)
    assert not any(True for _ in m.parameters()), "W must be a buffer, not a parameter"

def test_time_embedding():
    m = TimeEmbedding(64, 256, 1.0).to(DEVICE)
    out = m(torch.rand(4).to(DEVICE))
    assert out.shape == (4, 256)

def test_resblock_same_ch():
    m = ResBlock2D(32, 32, 256).to(DEVICE)
    x, t = torch.randn(2, 32, 4, 16).to(DEVICE), torch.randn(2, 256).to(DEVICE)
    assert m(x, t).shape == x.shape

def test_resblock_diff_ch():
    m = ResBlock2D(32, 64, 256).to(DEVICE)
    x, t = torch.randn(2, 32, 4, 16).to(DEVICE), torch.randn(2, 256).to(DEVICE)
    assert m(x, t).shape == (2, 64, 4, 16)

def test_downsample_even():
    m = DownsampleN(32).to(DEVICE)
    assert m(torch.randn(2, 32, 4, 16).to(DEVICE)).shape == (2, 32, 4, 8)

def test_downsample_odd():
    m = DownsampleN(32).to(DEVICE)
    out = m(torch.randn(2, 32, 4, 15).to(DEVICE))
    assert out.shape[-1] in (7, 8)

def test_upsample():
    m = UpsampleN(32).to(DEVICE)
    assert m(torch.randn(2, 32, 4, 8).to(DEVICE), 16).shape == (2, 32, 4, 16)

def test_self_attention():
    m = SelfAttention2D(64, heads=4).to(DEVICE)
    x = torch.randn(2, 64, 4, 32).to(DEVICE)
    assert m(x).shape == x.shape

def test_cross_attention():
    m = CrossAttention2D(64, heads=4).to(DEVICE)
    x    = torch.randn(2, 64, 4, 32).to(DEVICE)
    cond = torch.randn(2, 64, 1, 32).to(DEVICE)
    assert m(x, cond).shape == x.shape


# ── mult_channels tests ──────────────────────────────────────────────────────

def test_mult_channels_ascending():
    """Classic [1, 2, 4] — channels grow with depth."""
    model = UNet(base_channels=32, mult_channels=[1, 2, 4], time_dim=64).to(DEVICE)
    x = torch.randn(2, 2, 4, 32).to(DEVICE)
    t = torch.rand(2).to(DEVICE)
    out = model(x, t)
    assert out.shape == (2, 2, 3, 32), f"got {out.shape}"

def test_mult_channels_descending():
    """[4, 2, 1] — channels shrink with depth."""
    model = UNet(base_channels=16, mult_channels=[4, 2, 1], time_dim=64).to(DEVICE)
    x = torch.randn(2, 2, 4, 32).to(DEVICE)
    t = torch.rand(2).to(DEVICE)
    out = model(x, t)
    assert out.shape == (2, 2, 3, 32), f"got {out.shape}"

def test_mult_channels_arbitrary():
    """[2, 4, 1] — arbitrary order, as in the user's example."""
    model = UNet(base_channels=16, mult_channels=[2, 4, 1], time_dim=64).to(DEVICE)
    x = torch.randn(2, 2, 4, 32).to(DEVICE)
    t = torch.rand(2).to(DEVICE)
    out = model(x, t)
    assert out.shape == (2, 2, 3, 32), f"got {out.shape}"

def test_mult_channels_two_levels():
    """Only 2 levels."""
    model = UNet(base_channels=32, mult_channels=[1, 2], time_dim=64).to(DEVICE)
    x = torch.randn(2, 2, 3, 32).to(DEVICE)
    t = torch.rand(2).to(DEVICE)
    out = model(x, t)
    assert out.shape == (2, 2, 2, 32), f"got {out.shape}"

def test_mult_channels_four_levels():
    """4 levels."""
    model = UNet(base_channels=16, mult_channels=[1, 2, 4, 2], time_dim=64).to(DEVICE)
    x = torch.randn(2, 2, 5, 32).to(DEVICE)
    t = torch.rand(2).to(DEVICE)
    out = model(x, t)
    assert out.shape == (2, 2, 4, 32), f"got {out.shape}"

def test_mult_channels_constant():
    """All-same multiplier — flat channel count."""
    model = UNet(base_channels=32, mult_channels=[2, 2, 2], time_dim=64).to(DEVICE)
    x = torch.randn(2, 2, 4, 32).to(DEVICE)
    t = torch.rand(2).to(DEVICE)
    out = model(x, t)
    assert out.shape == (2, 2, 3, 32), f"got {out.shape}"

def test_mult_channels_backward_compat():
    """Omitting mult_channels keeps the original 2**i behaviour."""
    model = UNet(base_channels=16, depth=3, time_dim=64).to(DEVICE)
    x = torch.randn(2, 2, 4, 32).to(DEVICE)
    t = torch.rand(2).to(DEVICE)
    out = model(x, t)
    assert out.shape == (2, 2, 3, 32), f"got {out.shape}"


# ── full UNet tests ──────────────────────────────────────────────────────────

def test_unet_variable_n():
    model = UNet(base_channels=16, mult_channels=[1, 2], time_dim=64).to(DEVICE)
    for N in [17, 33, 64, 100, 128]:
        x = torch.randn(1, 2, 3, N).to(DEVICE)
        out = model(x, torch.rand(1).to(DEVICE))
        assert out.shape[-1] == N, f"N={N}: got {out.shape}"

def test_unet_no_nan():
    model = UNet(base_channels=16, mult_channels=[1, 2, 4], time_dim=64).to(DEVICE)
    x = torch.randn(2, 2, 4, 32).to(DEVICE)
    out = model(x, torch.rand(2).to(DEVICE))
    assert not torch.isnan(out).any() and not torch.isinf(out).any()

def test_unet_backward():
    model = UNet(base_channels=16, mult_channels=[1, 2, 4], time_dim=64).to(DEVICE)
    x = torch.randn(2, 2, 4, 32).to(DEVICE)
    model(x, torch.rand(2).to(DEVICE)).mean().backward()
    no_grad = [n for n, p in model.named_parameters() if p.grad is None]
    assert not no_grad, f"Params with no grad: {no_grad}"

def test_unet_train_eval():
    model = UNet(base_channels=16, mult_channels=[1, 2], time_dim=64, drop_path_rate=0.5).to(DEVICE)
    x, t = torch.randn(4, 2, 3, 32).to(DEVICE), torch.rand(4).to(DEVICE)
    model.eval()
    with torch.no_grad():
        assert torch.allclose(model(x, t), model(x, t)), "Eval not deterministic"

def test_unet_no_attention():
    model = UNet(base_channels=16, mult_channels=[1, 2], time_dim=64, use_attention=False).to(DEVICE)
    x = torch.randn(2, 2, 3, 32).to(DEVICE)
    out = model(x, torch.rand(2).to(DEVICE))
    assert out.shape == (2, 2, 2, 32)

def test_unet_param_count():
    model = UNet(base_channels=64, mult_channels=[1, 2, 4], time_dim=256)
    total = sum(p.numel() for p in model.parameters())
    print(f"       total params: {total:,}")
    assert total > 0


# ── run ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print(f"\nDevice: {DEVICE}\n{'─'*55}")

    run("GaussianFourierProjection",        test_fourier_projection)
    run("TimeEmbedding",                    test_time_embedding)
    run("ResBlock2D same channels",         test_resblock_same_ch)
    run("ResBlock2D channel change",        test_resblock_diff_ch)
    run("DownsampleN even N",              test_downsample_even)
    run("DownsampleN odd N",               test_downsample_odd)
    run("UpsampleN",                        test_upsample)
    run("SelfAttention2D",                  test_self_attention)
    run("CrossAttention2D",                 test_cross_attention)
    print()
    run("mult_channels=[1,2,4] ascending",  test_mult_channels_ascending)
    run("mult_channels=[4,2,1] descending", test_mult_channels_descending)
    run("mult_channels=[2,4,1] arbitrary",  test_mult_channels_arbitrary)
    run("mult_channels=2 levels",           test_mult_channels_two_levels)
    run("mult_channels=4 levels",           test_mult_channels_four_levels)
    run("mult_channels=[2,2,2] constant",   test_mult_channels_constant)
    run("mult_channels=None (compat)",      test_mult_channels_backward_compat)
    print()
    run("Variable N",                       test_unet_variable_n)
    run("No NaN / Inf",                     test_unet_no_nan)
    run("Backward pass",                    test_unet_backward)
    run("Train vs eval (DropPath)",         test_unet_train_eval)
    run("use_attention=False",              test_unet_no_attention)
    run("Param count",                      test_unet_param_count)

    print(f"\n{'─'*55}")
    print(f"Results: {len(PASSED)} passed, {len(FAILED)} failed")
    if FAILED:
        print(f"Failed:  {FAILED}")
        sys.exit(1)
    else:
        print("All tests passed ✓")


Device: cuda
───────────────────────────────────────────────────────
[PASS] GaussianFourierProjection
[PASS] TimeEmbedding
[PASS] ResBlock2D same channels
[PASS] ResBlock2D channel change
[PASS] DownsampleN even N
[PASS] DownsampleN odd N
[PASS] UpsampleN
[PASS] SelfAttention2D
[PASS] CrossAttention2D

[PASS] mult_channels=[1,2,4] ascending
[PASS] mult_channels=[4,2,1] descending
[PASS] mult_channels=[2,4,1] arbitrary
[PASS] mult_channels=2 levels
[PASS] mult_channels=4 levels
[PASS] mult_channels=[2,2,2] constant
[PASS] mult_channels=None (compat)

[PASS] Variable N
[PASS] No NaN / Inf
[PASS] Backward pass
[PASS] Train vs eval (DropPath)
[PASS] use_attention=False
       total params: 3,060,930
[PASS] Param count

───────────────────────────────────────────────────────
Results: 22 passed, 0 failed
All tests passed ✓
